# Video Player Challenge: real speed + pause



## 1. Install OpenCV
`imageio-ffmpeg` brings its own copy of ffmpeg, which Challenge 3 uses to make browser-playable (H.264) copies.

In [ ]:
%pip install opencv-python numpy matplotlib imageio-ffmpeg

## 2. Get the video
Downloads `race_car_slow_motion.mp4` from the course repo (Mod4) into this notebook.

In [ ]:
import os
import urllib.request

source = 'race_car_slow_motion.mp4'
url = 'https://raw.githubusercontent.com/fenago/cv2/main/Mod4/' + source

if not os.path.exists(source):
    urllib.request.urlretrieve(url, source)
print(source, 'is here:', os.path.exists(source))

# CHALLENGE #1
### Turns the frame stepper into a real player:
- plays at roughly the video's true speed (from its frame rate),
- **p** pauses on the current frame, **p** again resumes,
- **q**, **Q** or **Esc** quits.


## 3. Open the video and get its frame rate

In [ ]:
import cv2

# Use source = 0 to specify the web cam as the video source, OR
# specify the pathname to a video file to read.
source = 'race_car_slow_motion.mp4'  # if source = 0 then the web cam will be used as the video source

# Create a video capture object from the VideoCapture Class.
video_cap = cv2.VideoCapture(source)
print('Opened:', video_cap.isOpened())

# Get the frame rate so the video plays at roughly its true speed.
# get() always returns a float; some files and cameras report 0, so fall back to 30.
fps = video_cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

# Milliseconds to show each frame: 1000 ms / frames per second.
# For 29.97 FPS that is about 33 ms.
delay = int(1000 / fps)
print(f'FPS: {fps:.2f}, waiting {delay} ms per frame')

## 4. Play, pause, quit
A window opens. **p** = pause/resume, **q / Q / Esc** = quit. Click the video window first so it gets your key presses.

In [ ]:
# Create a named window for the video display.
win_name = 'Video Player'
cv2.namedWindow(win_name)

# Every key on the keyboard has a number. ord() looks it up.
# q = 113, Q = 81, Esc = 27, p = 112, P = 80.
quit_keys = (ord('q'), ord('Q'), 27)

# Enter a while loop to read and display the video frames one at a time.
while True:
    # Read one frame at a time using the video capture object.
    has_frame, frame = video_cap.read()
    if not has_frame:
        break

    # Display the current frame in the named window.
    cv2.imshow(win_name, frame)

    # Use the waitKey() function to monitor the keyboard for user input.
    # key = cv2.waitKey(0) will display the window indefinitely until any key is pressed.
    # key = cv2.waitKey(delay) shows the frame for one frame's worth of time,
    # then automatically advances to the next frame, so the video plays.
    key = cv2.waitKey(delay)

    # The return value of the waitKey() function indicates which key was pressed.
    # You can use this feature to check if the user selected the `q` key to quit the video stream.
    if key in quit_keys:
        # Exit the loop.
        break

    # Pause: a pause is just a waitKey(0) that only happens when the user asks for it.
    if key == ord('p') or key == ord('P'):
        while True:
            # Wait indefinitely on the current frame for the next key press.
            key = cv2.waitKey(0)
            # p resumes; a quit key exits; any other key is ignored and stays paused.
            if key == ord('p') or key == ord('P') or key in quit_keys:
                break
        # If a quit key was pressed while paused, exit the player too.
        if key in quit_keys:
            break

# Release the video capture object and close the window.
video_cap.release()
cv2.destroyWindow(win_name)
cv2.waitKey(1)  # lets the window actually close (needed on macOS and some Jupyter setups)

# Challenge 2: grayscale, half speed, banner from Frame: 1

Make a **grayscale** version of the race-car video, slowed to **half speed**, with a banner reading `Frame:` and the frame number. Save it as `race_car_gray.avi`. The **first frame must be included** and labeled `Frame: 1`.


## 2.1 Get race_car.mp4
The normal-speed clip from the same Mod4 folder.

In [ ]:
gray_source = 'race_car.mp4'
gray_url = 'https://raw.githubusercontent.com/fenago/cv2/main/Mod4/' + gray_source

if not os.path.exists(gray_source):
    urllib.request.urlretrieve(gray_url, gray_source)
print(gray_source, 'is here:', os.path.exists(gray_source))

## 2.2 A banner that works on grayscale frames


In [ ]:
def drawBannerTextGray(frame, text, banner_height_percent = 0.05, font_scale = 1.5,
                       text_value = 255, font_thickness = 2):
    # Draw a black filled banner across the top of a single-channel (grayscale) frame.
    banner_height = int(banner_height_percent * frame.shape[0])
    cv2.rectangle(frame, (0, 0), (frame.shape[1], banner_height), 0, thickness = -1)

    # Draw the text in white (one brightness value, not a BGR colour).
    left_offset = 20
    location = (left_offset, int(5 + banner_height / 2) + 10)
    cv2.putText(frame, text, location, cv2.FONT_HERSHEY_PLAIN, font_scale, text_value,
                font_thickness, cv2.LINE_AA)

## 2.3 Write race_car_gray.avi

In [ ]:
# Hint 1: a FRESH capture, so the first read() returns frame 1.
gray_cap = cv2.VideoCapture(gray_source)
print('Opened:', gray_cap.isOpened())

frame_w = int(gray_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(gray_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
src_fps = gray_cap.get(cv2.CAP_PROP_FPS)

# Half speed: same frames, half the playback rate (30 -> 15).
half_fps = src_fps / 2

# Hint 2: isColor = False because every frame we write has one channel.
# Size is (width, height); a frame of any other size is silently skipped.
gray_out = cv2.VideoWriter('race_car_gray.avi', cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'),
                           half_fps, (frame_w, frame_h), isColor = False)
print('Writer opened:', gray_out.isOpened())

frame_count = 0
while True:
    ok, frame = gray_cap.read()
    if not ok:
        break
    frame_count += 1                                   # 1 for the first frame

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)     # 3 channels -> 1 channel
    drawBannerTextGray(gray, 'Frame: ' + str(frame_count))
    gray_out.write(gray)

# Release both, or the file is not finished.
gray_cap.release()
gray_out.release()
print(f'Wrote {frame_count} frames at {half_fps:.1f} FPS = {frame_count / half_fps:.1f} seconds')

## 2.4 Check it: count the frames and look at frame 1


In [ ]:
import matplotlib.pyplot as plt

check = cv2.VideoCapture('race_car_gray.avi')
print('FPS in file:', check.get(cv2.CAP_PROP_FPS))

ok, first = check.read()
count = 1 if ok else 0
while check.read()[0]:
    count += 1
check.release()
print('Frames in file:', count)

plt.figure(figsize = (10, 6))
plt.imshow(first[:, :, ::-1])
plt.title('First frame of race_car_gray.avi')
plt.axis('off')
plt.show()

# Challenge 3: my own video (falls.mp4)

Every frame of my waterfall clip gets my **name** and a **timestamp** (that frame's time in the source). The same frames are written three times, **slow**, **normal** and **fast**, and then each version is **played** two ways: inside the notebook and in an OpenCV window.

The slowness and fastness live only in the FPS number written into each file, as in Lab 4: the frames are identical.

## 3.1 Settings and the stamp function
My video, `falls_1080p.mp4`, downloads from my GitHub repo ([saudie72/video-processing](https://github.com/saudie72/video-processing)) the first time this runs. It is a 1080p copy of my 4K original (190 MB, too big for GitHub). To use the original, put `falls.mp4` in this folder and set `my_source = 'falls.mp4'`.

In [ ]:
import os
import urllib.request
import cv2
import matplotlib.pyplot as plt

# ---- Settings: change these ----
# My video is in my GitHub repo; it downloads into this folder the first time.
my_url     = 'https://raw.githubusercontent.com/saudie72/video-processing/main/falls_1080p.mp4'
my_source  = 'falls_1080p.mp4'     # local copy of my video, or 0 for the webcam
my_name    = 'Saudie Feliz'        # text stamped on every frame
# SPEED: slow = 0.5 (half speed), normal = 1.0, fast = 2.0 (double speed)
speeds     = {'slow': 0.5, 'normal': 1.0, 'fast': 2.0}
out_width  = 1280                  # output width in pixels; frames are resized to this (keeps files small)
max_frames = 300                   # safety limit for the webcam (it never ends)

if my_source != 0 and not os.path.exists(my_source):
    urllib.request.urlretrieve(my_url, my_source)
print(my_source, 'is here:', my_source == 0 or os.path.exists(my_source))

def stamp(frame, name, seconds):
    # Black banner across the bottom with my name on the left and the time on the right.
    h, w = frame.shape[:2]
    bh = int(0.08 * h)
    cv2.rectangle(frame, (0, h - bh), (w, h), (0, 0, 0), thickness = -1)
    scale = bh / 40                # text size follows the frame size
    y = h - bh // 3
    cv2.putText(frame, name, (20, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2, cv2.LINE_AA)
    # TIMESTAMP: turn seconds into MM:SS.ss (e.g. 00:12.47) and draw it on the right
    t = f'{int(seconds // 60):02d}:{seconds % 60:05.2f}'
    (tw, _), _ = cv2.getTextSize(t, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.putText(frame, t, (w - tw - 20, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 255, 255), 2, cv2.LINE_AA)

## 3.2 Write the slow, normal and fast versions
One pass over the video: each frame is resized, stamped, and written to all three files. Takes about a minute for a 30-second clip.

In [ ]:
my_cap = cv2.VideoCapture(my_source)
print('Opened:', my_cap.isOpened())

src_w = int(my_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h = int(my_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps   = my_cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:                       # webcams often report 0
    fps = 30

# Output size: out_width wide, height scaled to keep the shape (kept even for video encoders).
w = min(out_width, src_w)
h = int(round(src_h * w / src_w / 2)) * 2
print(f'Source {src_w} x {src_h} at {fps:.2f} FPS -> writing {w} x {h}')

# SPEED: one writer per speed. Same frames, different FPS:
#   slow   = fps * 0.5 -> 15 FPS, plays twice as long (slow down)
#   normal = fps * 1.0 -> 30 FPS
#   fast   = fps * 2.0 -> 60 FPS, plays half as long (speed up)
fourcc  = cv2.VideoWriter_fourcc(*'mp4v')
writers = {label: cv2.VideoWriter(f'falls_{label}.mp4', fourcc, fps * m, (w, h))
           for label, m in speeds.items()}
print('Writers opened:', {label: out.isOpened() for label, out in writers.items()})

n = 0
while n < max_frames or my_source != 0:
    ok, frame = my_cap.read()
    if not ok:
        break
    frame = cv2.resize(frame, (w, h))          # writer size is (width, height); other sizes are skipped
    # TIMESTAMP: frame number / fps = this frame's time in seconds, stamped with my name
    stamp(frame, my_name, n / fps)
    for out in writers.values():
        out.write(frame)
    if n == 0:
        first_stamped = frame.copy()
    n += 1

# Release everything, or the mp4 files have no index and will not open.
my_cap.release()
for out in writers.values():
    out.release()

for label, m in speeds.items():
    print(f'falls_{label}.mp4: {n} frames at {fps * m:.2f} FPS = {n / (fps * m):.1f} s')

if n:
    plt.figure(figsize = (10, 6))
    plt.imshow(first_stamped[:, :, ::-1]); plt.title('First stamped frame'); plt.axis('off'); plt.show()

## 3.3 Check each file: frames, FPS, length, and the timestamp near the end
Reads every file back with a fresh capture, the way another program would.

In [ ]:
plt.figure(figsize = (18, 4))
for i, label in enumerate(speeds):
    path = f'falls_{label}.mp4'
    cap = cv2.VideoCapture(path)
    file_fps = cap.get(cv2.CAP_PROP_FPS)
    count, last = 0, None
    while True:
        ok, f = cap.read()
        if not ok:
            break
        count, last = count + 1, f
    cap.release()
    print(f'{path}: {count} frames, {file_fps:.2f} FPS, plays {count / file_fps:.1f} s')
    plt.subplot(1, 3, i + 1); plt.imshow(last[:, :, ::-1]); plt.title(f'{label}: last frame'); plt.axis('off')
plt.show()

## 3.4 Make browser-playable copies (H.264)
OpenCV wrote `mp4v`, which VLC and OpenCV play but a notebook's player (a browser) does not, the Lab 4 lesson. ffmpeg re-encodes each file to H.264. This uses ffmpeg if it is installed, otherwise the copy that came with `imageio-ffmpeg`.

In [ ]:
import shutil
import subprocess

ffmpeg = shutil.which('ffmpeg')
if ffmpeg is None:
    import imageio_ffmpeg                       # pip install imageio-ffmpeg
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
print('Using ffmpeg at:', ffmpeg)

for label in speeds:
    subprocess.run([ffmpeg, '-hide_banner', '-loglevel', 'error', '-y',
                    '-i', f'falls_{label}.mp4',
                    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '26',
                    f'falls_{label}_h264.mp4'], check = True)
    print(f'falls_{label}_h264.mp4', round(os.path.getsize(f'falls_{label}_h264.mp4') / 1e6, 1), 'MB')

## 3.5 Play all three inside the notebook
Slow plays for twice as long as normal, fast for half. Watch the timestamp: it shows the *source* time, so all three end at the same stamp.

In [ ]:
from IPython.display import Video, display, Markdown

for label, m in speeds.items():
    display(Markdown(f'### {label.capitalize()} ({m}x)'))
    display(Video(f'falls_{label}_h264.mp4', embed = True, width = 640))

## 3.6 Play all three in an OpenCV window (run on my own computer)
Uses the Challenge 1 player on each file in turn, at that file's own FPS: 15 for slow, 30 for normal, 60 for fast.

Keys (click the window first): **p** = pause / resume, **q** or **Q** = skip to the next version, **Esc** = stop.

In [ ]:
def play(path, win_name):
    # Play one file at its own frame rate. Returns False if Esc was pressed.
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30
    # SPEED: each file plays at its own FPS, so slow waits longer per frame and fast waits less
    delay = max(1, int(1000 / fps))            # ms per frame: 15 FPS = 66 ms, 30 = 33 ms, 60 = 16 ms
    keep_going = True
    while True:
        has_frame, frame = cap.read()
        if not has_frame:
            break
        cv2.imshow(win_name, frame)
        key = cv2.waitKey(delay)
        if key == 27:                          # Esc: stop everything
            keep_going = False
            break
        if key in (ord('q'), ord('Q')):        # q: skip to the next version
            break
        if key in (ord('p'), ord('P')):        # p: pause until p again (q / Esc still work)
            while True:
                key = cv2.waitKey(0)
                if key in (ord('p'), ord('P'), ord('q'), ord('Q'), 27):
                    break
            if key == 27:
                keep_going = False
                break
            if key in (ord('q'), ord('Q')):
                break
    cap.release()
    cv2.destroyWindow(win_name)
    cv2.waitKey(1)                             # lets the window close (macOS / some Jupyter setups)
    return keep_going

for label, m in speeds.items():
    print(f'Playing {label} ({m}x)...')
    if not play(f'falls_{label}.mp4', f'falls - {label} ({m}x)'):
        break
print('Done')

## Ideas to experiment with
- **Speed:** try `0.25` (very slow) or `4.0` in `speeds`. The file length changes; the frames do not.
- **Real time-lapse:** keep only every 3rd frame (`if n % 3 == 0:`) instead of raising the FPS, and compare.
- **Grayscale:** combine with Challenge 2 (`isColor = False`, a one-value text colour).
- **Logo:** `logo = cv2.imread('logo.png')`, resize it, then `frame[10:10+lh, 10:10+lw] = logo` before writing.
- **Webcam:** `my_source = 0`; `max_frames` stops it.